In [1]:
import boto3
import sagemaker
import pickle
import zarr
from zarr.codecs import ZstdCodec
from zarr.codecs.numcodecs import Zstd
import pandas
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
import s3fs
import os
import shutil
import kagglehub
from src.utils import *
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, message="numpy.core")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
%store -r

In [3]:
session = boto3.Session()
region = session.region_name
s3 = session.client(service_name="s3", region_name=region)
default_bucket = default_bucket
s3_project_root = s3_project_root

# Initial Dataset Download

In [5]:
# Download latest version
path = kagglehub.dataset_download("mohamedalqblawi/rplan-pickle-files")
print("Path to dataset files:", path)

Path to dataset files: /home/sagemaker-user/.cache/kagglehub/datasets/mohamedalqblawi/rplan-pickle-files/versions/1


# Deserializing and Uploading to S3

In [4]:
# Deserializing settings
BATCH_SIZE = 640
COMPRESSORS = Zstd()
LOCAL_DATA_DIR = Path("../.cache/kagglehub/datasets/mohamedalqblawi/rplan-pickle-files/versions/1/pickle/")
ZARR_REF = "data.zarr"
METADATA_REF = "metadata.parquet"

/opt/conda/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:141: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


In [ ]:
bad_pickles = ['../.cache/kagglehub/datasets/mohamedalqblawi/rplan-pickle-files/versions/1/pickle/train/28830.pkl',
               '../.cache/kagglehub/datasets/mohamedalqblawi/rplan-pickle-files/versions/1/pickle/train/74757.pkl',
               '../.cache/kagglehub/datasets/mohamedalqblawi/rplan-pickle-files/versions/1/pickle/train/80704.pkl']

pickles = list(LOCAL_DATA_DIR.rglob("*.pkl"))
pickles = [p for p in pickles if p not in bad_pickles]
store = zarr.open(ZARR_REF, mode="w", zarr_version=2, zarr_format=2)

# batch data
inside_masks, boundary_masks, room_masks, door_masks, meta_batch = [], [], [], [], []
edge_index_batch, edge_attr_batch, node_features_batch, node_labels_batch = [], [], [], []
node_total_batch, edge_total_batch = [], []

# zarr datasets, initialized on first write
inside_masks_ds, boundary_masks_ds, room_masks_ds, door_masks_ds = None, None, None, None
edge_index_ds, edge_attr_ds, node_features_ds, node_labels_ds = None, None, None, None
node_total_ds, edge_total_ds = None, None

sample_counter = 0
room_metadata = []

MAX_NODES = 24
MAX_EDGES = 48

def pad_graph(node_features, node_labels, edge_index, edge_attr, max_nodes, max_edges):
    """Returns padded arrays + true lengths."""
    n = len(node_labels)
    e = edge_index.shape[1]  # edge_index shape: [2, E]

    nf_pad = np.zeros((max_nodes, node_features.shape[1]), dtype=node_features.dtype)
    nl_pad = np.full(max_nodes, -1, dtype=node_labels.dtype)  # -1 = padding sentinel
    ei_pad = np.zeros((2, max_edges), dtype=edge_index.dtype)
    ea_pad = np.zeros((2, max_edges), dtype=edge_attr.dtype)

    nf_pad[:n] = node_features
    nl_pad[:n] = node_labels
    ei_pad[:, :e] = edge_index
    ea_pad[:, :e] = edge_attr.T

    return nf_pad, nl_pad, ei_pad, ea_pad, n, e

def flush_batch(im_batch, bm_batch, rm_batch, dr_batch, ei_batch, ea_batch, nf_batch, 
                nl_batch,nt_batch, et_batch):
    global inside_masks_ds, boundary_masks_ds, room_masks_ds, door_masks_ds
    global edge_index_ds, edge_attr_ds, node_features_ds, node_labels_ds
    global node_total_ds, edge_total_ds

    im_arr = np.stack(im_batch)
    bm_arr = np.stack(bm_batch)
    rm_arr = np.stack(rm_batch)
    dr_arr = np.stack(dr_batch)
    edge_arr = np.stack(ei_batch)
    edge_attr_arr =np.stack(ea_batch)
    node_feature_arr = np.stack(nf_batch)
    node_label_arr = np.stack(nl_batch)
    node_total_arr = np.stack(nt_batch)
    edge_total_arr = np.stack(et_batch)

    if inside_masks_ds is None:
        # First flush — create arrays with explicit shape/dtype, data as keyword arg
        inside_masks_ds = store.create_array(
            "inside_masks",
            chunks=(BATCH_SIZE, *im_arr.shape[1:]),
            data=im_arr
        )
        boundary_masks_ds = store.create_array(
            "boundary_masks",
            chunks=(BATCH_SIZE, *bm_arr.shape[1:]),
            data=bm_arr
        )
        room_masks_ds = store.create_array(
            "room_masks",
            chunks=(BATCH_SIZE, *rm_arr.shape[1:]),
            data=rm_arr
        )
        door_masks_ds = store.create_array(
            "door_masks",
            chunks=(BATCH_SIZE, *dr_arr.shape[1:]),
            data=dr_arr
        )
        edge_index_ds = store.create_array(
            "edge_index",
            chunks=(BATCH_SIZE, *edge_arr.shape[1:]),
            data=edge_arr
        )
        edge_attr_ds = store.create_array(
            "edge_attr",
            chunks=(BATCH_SIZE, *edge_attr_arr.shape[1:]),
            data=edge_attr_arr
        )
        node_features_ds = store.create_array(
            "node_features",
            chunks=(BATCH_SIZE, *node_feature_arr.shape[1:]),
            data=node_feature_arr
        )
        node_labels_ds = store.create_array(
            "node_labels",
            chunks=(BATCH_SIZE, *node_label_arr.shape[1:]),
            data=node_label_arr
        )
        node_total_ds = store.create_array(
            "node_totals",
            chunks=(BATCH_SIZE, *node_total_arr.shape[1:]),
            data=node_total_arr
        )
        edge_total_ds = store.create_array(
            "edge_totals",
            chunks=(BATCH_SIZE, *edge_total_arr.shape[1:]),
            data=edge_total_arr
        )

    else:
        # Subsequent flushes — append along axis 0
        inside_masks_ds.append(im_arr)
        boundary_masks_ds.append(bm_arr)
        room_masks_ds.append(rm_arr)
        door_masks_ds.append(dr_arr)
        edge_index_ds.append(edge_arr)
        edge_attr_ds.append(edge_attr_arr)
        node_features_ds.append(node_feature_arr)
        node_labels_ds.append(node_label_arr)
        node_total_ds.append(node_total_arr)
        edge_total_ds.append(edge_total_arr)

for pkl_path in tqdm(pickles, "Deserializing pickles...", len(pickles), unit="pickles"):
    with open(pkl_path, "rb") as f:
        i_mask, b_mask, r_mask, d_mask, meta = pickle.load(f)
    
    b_mask = b_mask//127
        
    inside_masks.append(i_mask)
    boundary_masks.append(b_mask) # original data encodes walls as 127, doors as 255
    room_masks.append(r_mask)
    door_masks.append(d_mask)
    meta_batch.append(meta)
    
    # Graph extraction
    stacked_layers = stack_normalize(b_mask, r_mask, d_mask)
    num_labels, labels, stats, centroids = conn_components(
        i_mask, b_mask, r_mask, d_mask
        )
    rwb = rooms_with_bounds(stacked_layers, labels)
    adj_graph = extract_all_adjacencies(rwb)
    src, dst, attrs = edge_arrays(adj_graph)
    
    edge_index = np.array([src, dst])
    edge_attrs = np.array(attrs)
    node_features = np.array(node_array(centroids[1:, :], stats[1:, :]))
    node_labels = np.array(join_meta_category(centroids[1:, :], meta))
    
    nf_pad, nl_pad, ei_pad, ea_pad, n, e = pad_graph(node_features, node_labels, 
                                                     edge_index, edge_attrs, MAX_NODES, MAX_EDGES)
    
    edge_index_batch.append(ei_pad)
    edge_attr_batch.append(ea_pad)
    node_features_batch.append(nf_pad)
    node_labels_batch.append(nl_pad)
    node_total_batch.append([n])
    edge_total_batch.append([e])
    
    
    sample_counter += 1

    if len(inside_masks) >= BATCH_SIZE:
        flush_batch(inside_masks, boundary_masks, room_masks, door_masks, 
                    edge_index_batch, edge_attr_batch, node_features_batch, node_labels_batch,
                    node_total_batch, edge_total_batch)
        room_metadata.extend(meta_batch)
        inside_masks, boundary_masks, room_masks, door_masks, meta_batch = [], [], [], [], []
        edge_index_batch, edge_attr_batch, node_features_batch, node_labels_batch = [], [], [], []
        node_total_batch, edge_total_batch = [], []
  
# Flush any remaining samples that didn't fill a full batch
if inside_masks:
    flush_batch(inside_masks, boundary_masks, room_masks, door_masks, 
                edge_index_batch, edge_attr_batch, node_features_batch, node_labels_batch,
                node_total_batch, edge_total_batch)
    room_metadata.extend(meta_batch)

# writing metadata
df = pd.DataFrame(room_metadata)
df["id"] = [str(pkl).split("/")[-1] for pkl in pickles]
df.insert(0, "sample_id", [f"sample_{i:05d}" for i in range(sample_counter)])
df.to_parquet(METADATA_REF, index=False)

print(f"Done. {sample_counter} samples written.")

Deserializing pickles...: 100%|██████████| 700/700 [00:56<00:00, 12.37pickles/s]


Done. 700 samples written.


In [6]:
datastore_dest_path = f"{s3_project_root}/store"
metadata_dest_path = f"{s3_project_root}/metadata"

# Uses your AWS credentials automatically (~/.aws/credentials or env vars)
fs = s3fs.S3FileSystem()

# Upload metadata to S3
fs.put(METADATA_REF, f"{default_bucket}/{metadata_dest_path}/{METADATA_REF}")

# Upload data store
fs.put(ZARR_REF, f"{default_bucket}/{datastore_dest_path}/{ZARR_REF}", recursive=True)

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

# Clean up local

In [7]:
if Path(METADATA_REF).exists():
    os.remove(METADATA_REF)

if Path(ZARR_REF).exists():
    shutil.rmtree(ZARR_REF)